In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'http://localhost:5001/'
idp_name_1 = 'FakeCAS'
idp_username_1 = None
idp_password_1 = None
project_name = None
workflow_name_x = None
workflow_name_y = None
abort_repeat_count = 25
start_y_repeat_count = 25
start_form_content = 'Test request content'
abort_reason = 'Aborted by automated test'
default_result_path = None
close_on_fail = False
transition_timeout = 30000
browser_type = 'chromium'
ignore_https_errors = False

In [ ]:
import tempfile

if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')

if project_name is None:
    project_name = datetime.now().strftime('TEST-WF-CUMULATIVE-%Y%m%d%H%M')
if workflow_name_x is None:
    workflow_name_x = input(prompt='Workflow template name X (simple-approval)')
if workflow_name_y is None:
    workflow_name_y = input(prompt='Workflow template name Y (example-wizard)')

# ワークフローの累積実行

- サブシステム名: アドオン
- ページ/アドオン: Workflow
- 機能分類: ワークフロー累積実行（中断と開始の繰り返しによるUI整合性検証）
- シナリオ名: Xの開始と中断を繰り返したのちYの開始を繰り返し、最後に改めてXを開始してタスク表示が正しく反映されることを確認する
- 用意するテストデータ: アカウント、ワークフローテンプレート名 X(simple-approval) / Y(example-wizard)
- 事前条件: X/Y のテンプレート登録は事前に完了していること

In [ ]:
work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

In [ ]:
import asyncio
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path, ignore_https_errors=ignore_https_errors)

## 新規タブを開き、『同意する』をクリックする

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    await page.locator('//button[text() = "同意する"]').click()
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step, new_page=True)

## 『ユーザー名』『パスワード』を入力し『ログイン』をクリックする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## 『新規プロジェクト作成』をクリックしてプロジェクトを作成する

プロジェクト一覧に当該プロジェクト名が表示されること

In [ ]:
async def _step(page):
    await expect(page.locator('//*[@data-test-create-project-modal-button]')).to_have_count(1)
    await grdm.ensure_project_exists(page, project_name, transition_timeout=transition_timeout)

await run_pw(_step)

## 作成したプロジェクトをクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
project_url = None

async def _step(page):
    global project_url
    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]').click()
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    project_url = page.url

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから『アドオン』をクリックする

アドオン設定画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "アドオン"]').click()
    await expect(page.locator('//h3[text() = "アドオンを選択"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 『アドオンを選択』パネル内『Workflow』行の『有効にする』をクリックする

『Workflowアドオン規約』ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('//div[@full_name = "Workflow"]//a[text() = "有効にする"]').click()
    await expect(page.locator('//button[@data-bb-handler = "confirm"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 『確認』をクリックする

『アドオンを構成』パネル内に『Workflow』の行が追加されること

In [ ]:
async def _step(page):
    await page.locator('//button[@data-bb-handler = "confirm"]').click()
    await expect(page.locator('#activate-workflow-select')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 『ワークフローテンプレート』ドロップダウンからテンプレートXを選択する

テンプレートXが選択されること

In [ ]:
async def _step(page):
    await page.locator('#activate-workflow-select').scroll_into_view_if_needed()
    option = page.locator(f'#activate-workflow-select option:has-text("{workflow_name_x}")')
    value = await option.get_attribute('value')
    await page.locator('#activate-workflow-select').select_option(value=value)

await run_pw(_step)

## 『有効化』をクリックする

『ワークフロー権限の付与』ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('#activationsPanel').locator('//button[.//span[contains(text(), "有効化")]]').click()
    await expect(page.locator('#activateTokenPermissionModal').locator('//h4[contains(text(), "ワークフロー権限の付与")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 『権限を付与して有効化』をクリックする

有効化済みテンプレート一覧にテンプレートXが表示されること

In [ ]:
async def _step(page):
    await page.locator('//button[contains(text(), "権限を付与して有効化")]').click()
    await expect(page.locator('#activationsPanel').locator(f'//td//strong[text()="{workflow_name_x}"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 『ワークフローテンプレート』ドロップダウンからテンプレートYを選択する

テンプレートYが選択されること

In [ ]:
async def _step(page):
    await page.locator('#activate-workflow-select').scroll_into_view_if_needed()
    option = page.locator(f'#activate-workflow-select option:has-text("{workflow_name_y}")')
    value = await option.get_attribute('value')
    await page.locator('#activate-workflow-select').select_option(value=value)

await run_pw(_step)

## 『有効化』をクリックする

『ワークフロー権限の付与』ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('#activationsPanel').locator('//button[.//span[contains(text(), "有効化")]]').click()
    await expect(page.locator('#activateTokenPermissionModal').locator('//h4[contains(text(), "ワークフロー権限の付与")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 『権限を付与して有効化』をクリックする

有効化済みテンプレート一覧にテンプレートXとテンプレートYの両方が表示されること

In [ ]:
async def _step(page):
    await page.locator('//button[contains(text(), "権限を付与して有効化")]').click()
    await expect(page.locator('#activationsPanel').locator(f'//td//strong[text()="{workflow_name_x}"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#activationsPanel').locator(f'//td//strong[text()="{workflow_name_y}"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトURL/workflow を開き、『実行』タブを選択する

ワークフローコンソールが表示され、『実行』タブのワークフロー選択ドロップダウンにテンプレートXとテンプレートYが含まれること

In [ ]:
async def _step(page):
    await page.goto(project_url.rstrip('/') + '/workflow')
    await page.get_by_role('tab', name='実行', exact=True).click()
    await expect(page.locator('#workflow-template-select')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator(f'#workflow-template-select option:has-text("{workflow_name_x}")')).to_have_count(1, timeout=transition_timeout)
    await expect(page.locator(f'#workflow-template-select option:has-text("{workflow_name_y}")')).to_have_count(1, timeout=transition_timeout)

await run_pw(_step)

## 『実行履歴』タブと『タスク』タブの『表示件数』を100に変更する

以降の累積検証で25件超を扱うため、両タブの『表示件数』プルダウンが100に切り替わること

In [ ]:
async def _step(page):
    # 「タスク」タブは子要素にバッジが入るため has_text で参照する
    task_tab = page.locator('[role="tab"]').filter(has_text='タスク')
    history_tab = page.get_by_role('tab', name='実行履歴', exact=True)
    for tab in [history_tab, task_tab]:
        await tab.click()
        # アクティブなタブパネル内に表示件数の select は1つだけ存在する
        page_size_select = page.locator('select').filter(has=page.locator('option[value="100"]'))
        await page_size_select.select_option(value='100')
        await expect(page_size_select).to_have_value('100', timeout=transition_timeout)

await run_pw(_step)


## 各回について、『実行』タブでテンプレートXを選択→申請フォームに入力→『ワークフローを開始』→『タスク』タブで『Review Application』タスクが出現するまで『更新』して待機→『実行履歴』タブで新規プロセスを『キャンセル』→『キャンセルを実行』する、を abort_repeat_count 回繰り返す

各回において、Review Applicationタスクの出現（PT1Sタイマー+HTTPサービスタスク完了でプロセスが安定状態に到達）を確認したうえで中断ダイアログが閉じ、runningプロセスが一覧から消えること（「完了した実行を非表示」フィルタで非表示化）。全回完了後、『タスク』タブにテンプレートX由来の『Review Application』タスクが1件も表示されないこと

In [ ]:
async def _step(page):
    # 「タスク」タブは件数バッジで accessible name が変動するため has_text で参照
    task_tab = page.locator('[role="tab"]').filter(has_text='タスク')
    exec_tab = page.get_by_role('tab', name='実行', exact=True)
    history_tab = page.get_by_role('tab', name='実行履歴', exact=True)
    running_rows = page.locator('tr[data-test-workflow-run-row]')
    refresh_btn = page.get_by_role('button', name='更新')
    x_task_rows = page.locator('tr[data-test-workflow-task-row]:has(strong:text-is("Review Application"))')
    dlg = page.locator('.workflow-cancel-dialog')

    for i in range(abort_repeat_count):
        print(f'[X abort {i+1}/{abort_repeat_count}]')

        # Xを開始
        await exec_tab.click()
        option = page.locator(f'#workflow-template-select option:has-text("{workflow_name_x}")')
        value = await option.get_attribute('value')
        await page.locator('#workflow-template-select').select_option(value=value)
        await page.locator('#workflow-field-confirm_checkbox').check()
        await page.locator('#workflow-field-request_content').fill(f'{start_form_content} - X#{i+1}')
        await page.locator('button[data-test-workflow-start-button]').click()

        # Review Applicationタスクが出現するまで『更新』で再取得しつつ待つ
        # （UIは自動ポーリングしないので、明示的に再取得する。
        #  また不安定状態（タイマー中/HTTPサービスタスク中）でキャンセルするとgatewayエラーになるため、
        #  reviewTask出現を安定状態の到達シグナルとして利用する）
        await task_tab.click()
        for _ in range(transition_timeout // 3000):
            await asyncio.sleep(2)
            await refresh_btn.click()
            if await x_task_rows.count() >= 1:
                break
        await expect(x_task_rows).to_have_count(1, timeout=transition_timeout)

        # プロセスをキャンセル
        await history_tab.click()
        await running_rows.locator('button[data-test-workflow-run-cancel]').click()
        await dlg.locator('#cancel-reason').fill(abort_reason)
        await dlg.locator('.modal-footer button.btn-danger').click()

        # キャンセル成功: ダイアログが閉じ、「完了した実行を非表示」フィルタでrunning行が消える
        await expect(dlg).to_have_count(0, timeout=transition_timeout)
        await expect(running_rows).to_have_count(0, timeout=transition_timeout)

    # 全回完了後、タスクタブにXのタスクが無いこと（キャンセルによりタスクも完了扱い→フィルタで非表示）
    await task_tab.click()
    await refresh_btn.click()
    await expect(x_task_rows).to_have_count(0, timeout=transition_timeout)
    print(f'X abort loop completed: {abort_repeat_count} processes cancelled, 0 Review Application tasks')

await run_pw(_step)


## 各回について、『実行』タブでテンプレートYを選択→『ワークフローを開始』をクリックする、を start_y_repeat_count 回繰り返す

各回において『ワークフローを開始』後、『タスク』タブに『Wizard Form Example』タスクが累積する（i 回目終了時に i 件）。全回完了後、『タスク』タブの『Wizard Form Example』タスクが start_y_repeat_count 件表示されること

In [ ]:
async def _step(page):
    task_tab = page.locator('[role="tab"]').filter(has_text='タスク')
    exec_tab = page.get_by_role('tab', name='実行', exact=True)
    y_task_rows = page.locator('tr[data-test-workflow-task-row]:has(strong:text-is("Wizard Form Example"))')
    template_select = page.locator('#workflow-template-select')

    for i in range(start_y_repeat_count):
        print(f'[Y start {i+1}/{start_y_repeat_count}]')

        # 直前iterationのstartクリックによる自動tasks遷移が遅延して、ここでのexec_tab.clickを
        # 上書きすることがある。実行タブ固有の #workflow-template-select が可視になるまで
        # click をリトライしてレースに勝つ
        for attempt in range(5):
            await exec_tab.click()
            await asyncio.sleep(1)
            if await template_select.is_visible():
                break
        else:
            raise AssertionError(f'exec_tab did not activate after 5 retries at Y iteration {i+1}')

        option = page.locator(f'#workflow-template-select option:has-text("{workflow_name_y}")')
        value = await option.get_attribute('value')
        await template_select.select_option(value=value)
        await page.locator('button[data-test-workflow-start-button]').click()

        # Yのタスクはプロセス起動と同時に生成され、タブ切替時のfetchで反映される
        await task_tab.click()
        await expect(y_task_rows).to_have_count(i + 1, timeout=transition_timeout)

    print(f'Y start loop completed: {start_y_repeat_count} Wizard Form Example tasks')

await run_pw(_step)


## 『実行』タブでテンプレートXを選択し申請フォームに入力して『ワークフローを開始』をクリックする

クリックが成功すること（起動成功の確認は次ステップのタスクタブで行う）

In [ ]:
async def _step(page):
    await page.get_by_role('tab', name='実行', exact=True).click()
    option = page.locator(f'#workflow-template-select option:has-text("{workflow_name_x}")')
    value = await option.get_attribute('value')
    await page.locator('#workflow-template-select').select_option(value=value)
    await page.locator('#workflow-field-confirm_checkbox').check()
    await page.locator('#workflow-field-request_content').fill(f'{start_form_content} - X final')
    await page.locator('button[data-test-workflow-start-button]').click()

await run_pw(_step)


## 『タスク』タブを開き『更新』をクリックしてテンプレートX由来の『Review Application』タスクが表示されていることを確認する

simple-approvalのreviewTaskはタイマー経過後に作成され、UIは自動ポーリングしないため『更新』で再取得する。『Review Application』タスクが1件以上表示され、かつ『Wizard Form Example』タスクが start_y_repeat_count 件のまま維持されていること

In [ ]:
async def _step(page):
    task_tab = page.locator('[role="tab"]').filter(has_text='タスク')
    await task_tab.click()

    # simple-approvalのreviewTaskはPT1Sタイマー経過後、HTTPサービスタスク経由で作成される。
    # UIはタブ切替時にfetchするが自動ポーリングはしないため、『更新』で明示的に再取得する。
    x_task_rows = page.locator('tr[data-test-workflow-task-row]:has(strong:text-is("Review Application"))')
    refresh_btn = page.get_by_role('button', name='更新')
    for _ in range(transition_timeout // 3000):
        await asyncio.sleep(2)
        await refresh_btn.click()
        if await x_task_rows.count() >= 1:
            break
    await expect(x_task_rows).to_have_count(1, timeout=transition_timeout)

    y_task_rows = page.locator('tr[data-test-workflow-task-row]:has(strong:text-is("Wizard Form Example"))')
    await expect(y_task_rows).to_have_count(start_y_repeat_count, timeout=transition_timeout)

await run_pw(_step)


## 後始末

ブラウザコンテキストが終了すること

In [ ]:
await finish_pw_context(screenshot=False, last_path=default_result_path)

In [ ]:
!rm -fr {work_dir}